# 08 · D/Live aus Spielerbewegung mit Machine Learning

**Frage:** Kann ein gelernter Spielstatus aus Spielerpositionen die Unterbrechungserkennung ergänzen?

Erster, bewusst kleiner Versuch nach Lang et al. (2022), [Predicting the in-game status in soccer with machine learning using spatiotemporal player tracking data](https://doi.org/10.1038/s41598-022-19948-1). Die Originalstudie nutzt 102 Spiele, geordnete Daten für 22 Spieler und 1320 Eingabewerte. Hier haben wir ein Spiel, lückenhafte wechselnde Track-Slots und nur fünf Minuten öffentliche Referenz. Wir testen deshalb eine **angepasste Methodik**, keine exakte Reproduktion und keine behauptete Übertragung der veröffentlichten 92 % Accuracy.

**Parallel zur Annotation:** Dieses Notebook liest und verändert weder Notebook 07 noch `segments.csv`, `reviewed_segments.csv` oder die Annotationsansicht. Es trainiert ausschließlich mit der öffentlichen Referenz. Vorhersagen für die drei Kontrollclips werden ohne deren Annotationen erzeugt und für einen späteren Vergleich gespeichert.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from importlib.metadata import version
import hashlib
import json
import sys
import xml.etree.ElementTree as ET

import joblib
import numpy as np
import pandas as pd
from scipy.ndimage import minimum_filter1d, median_filter
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display, Markdown

## 1. Festgelegter Versuchsplan

- Binäres Ziel: **D** = unterbrochen, **L** = H oder A im laufenden Spiel. Die Teamzuordnung wird hier nicht neu trainiert.
- Nur Spielermerkmale: Position, daraus abgeleitete Bewegung und Beobachtungsabdeckung. Kein Balltrack, kein Audio, keine Smart Tags und keine manuell gesetzten Kontrolllabels als Modellinput.
- 24 Merkmale für die sichtbare Gruppe und beide Teams, jeweils zu neun Zeitpunkten von −4 bis +4 s: **216 Eingabemerkmale** vor trainierten Missing-Indikatoren. Diese reduzierte Darstellung ist eine Anpassung an die kleinen und unvollständigen Daten.
- Logistische Regression als einfache Referenz und AdaBoost mit flachen Entscheidungsbäumen. Eine Konfiguration pro Modell, keine Suche und keine nachträgliche Schwellenanpassung. Klassifikation ab D-Score 0,5; dieser Score ist nicht als kalibrierte Wahrscheinlichkeit zu verstehen.
- Rohvorhersage und Medianfilter über **91 Frames / 3,64 s** werden getrennt berichtet. Der Filter stammt aus dem Paper und wird hier unverändert als Vergleich übernommen.
- Zeitlich getrennte Auswertung: vier Blöcke mit jeweils einer vollständigen öffentlichen D-Phase, zusätzlicher Abstand von zwölf Sekunden zu Trainingszentren. Keine zufällige Aufteilung benachbarter Frames.

Die vier D-Phasen sind extrem wenig unabhängige Ereignisse. Außerdem haben frühere Notebook-Versuche bereits diese öffentliche Referenz verwendet. Die Ergebnisse sind explorative interne Prüfungen innerhalb desselben Spiels; sie sind kein unabhängiger Matchtest wie in der Studie.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
ARCHIVE = PROJECT_DIR / "reference/mtah2027_official_3de6b60.zip"
OUTPUT_DIR = PROJECT_DIR / "output/player_status_ml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
CONTACT_MANIFEST = PROJECT_DIR / "artifacts/upstream/contact_classification/run_manifest.json"
BASELINE_PATHS = {
    "Kontakt 04": PROJECT_DIR / "artifacts/upstream/contact_classification/submissions/contact_state_shared_D.csv",
    "Heuristik 06": PROJECT_DIR / "artifacts/upstream/stoppage_detection/submissions/contact_state_stoppage_v1.csv",
}
SCENES_PATH = PROJECT_DIR / "artifacts/upstream/stoppage_detection/review_scenes.csv"
config = json.loads(CONTACT_MANIFEST.read_text())
FPS, N_FRAMES, PITCH = config["fps"], config["n_frames"], np.array(config["pitch_m"], dtype=np.float32)
HALVES = [("1. Halbzeit", 11185, 90651), ("2. Halbzeit", 110808, 190197)]
SEED, TRAIN_STRIDE = 42, 5  # Trainingszentren mit 5 Hz; Vorhersage und Prüfung mit 25 Hz.
TIME_OFFSETS_SECONDS = [-4, -2, -0.8, -0.4, 0, 0.4, 0.8, 2, 4]
DIFFERENCE_OFFSET = 5  # Geschwindigkeit über 0,4 s; Beschleunigung nochmals über 0,4 s.
MAX_SPEED_M_S, MAX_ACCELERATION_M_S2 = 15.0, 15.0  # Qualitätsgrenzen für diesen ersten Versuch.
MEDIAN_FRAMES, PURGE_FRAMES = 91, 12 * FPS
MAX_FEATURE_RADIUS = round(max(abs(s) for s in TIME_OFFSETS_SECONDS) * FPS) + 2 * DIFFERENCE_OFFSET
assert PURGE_FRAMES >= 2 * MAX_FEATURE_RADIUS + MEDIAN_FRAMES // 2, "Zeitliche Eingaben von Training und Prüfblock können sich überlappen."
MODEL_CONFIG = {
    "LogReg": {"C": 0.1, "solver": "lbfgs", "max_iter": 1000},
    "AdaBoost": {"n_estimators": 100, "learning_rate": 0.5, "max_depth": 1, "min_samples_leaf": 10},
}
PROTECTED_PATHS = [ARCHIVE, CONTACT_MANIFEST, *BASELINE_PATHS.values(), SCENES_PATH,
                   PROJECT_DIR / "artifacts/upstream/stoppage_detection/run_manifest.json"]
INPUT_HASHES = {str(p.relative_to(PROJECT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest() for p in PROTECTED_PATHS}
assert INPUT_HASHES[str(ARCHIVE.relative_to(PROJECT_DIR))] == config["archive_sha256"]
frame_ids = np.arange(N_FRAMES)
half_mask = np.zeros(N_FRAMES, dtype=bool)
for _, first, last in HALVES:
    half_mask[first:last + 1] = True
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False})
display(pd.Series({name: version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]}, name="Version").to_frame())
print(f"Maximaler Rohdatenkontext eines Merkmals: ±{MAX_FEATURE_RADIUS / FPS:.2f} s; Trainingsabstand zum Prüfblock: {PURGE_FRAMES / FPS:.1f} s.")
display(pd.DataFrame(MODEL_CONFIG).fillna(""))

## 2. Öffentliche Referenz und vollständige Unterbrechungen

Die Referenz enthält H/A/D nur im veröffentlichten Ausschnitt; X außerhalb wird nicht als Trainingslabel verwendet. Für die Blockbildung verwenden wir die D-Phasen der öffentlichen Referenz. Die Grenzen liegen jeweils in der Mitte zwischen benachbarten D-Phasen. Diese labelgestützte Blockbildung hält vollständige Unterbrechungen zusammen und ist Teil des offengelegten Versuchsdesigns, kein Modellmerkmal.

Beim Prüfen eines Blocks sieht das Modell weder dessen Zielwerte noch Zielwerte aus dem zusätzlichen Abstand. Zukunftskontext aus den **unbeschrifteten Spielerpositionen** ist erlaubt, da dieser Versuch wie das Paper offline arbeitet. Imputation, Skalierung und Klassifikator werden ausschließlich auf dem jeweiligen Training angepasst.

In [ ]:
with ZipFile(ARCHIVE) as official:
    archive_root = official.namelist()[0]
    gt = pd.read_csv(BytesIO(official.read(archive_root + "evaluation/gt/BAR-ATH_possession_gt_public.csv")))
assert np.array_equal(gt["frame"], frame_ids), "GT und Videoindex stimmen nicht überein."
public_mask = gt["label"].ne("X").to_numpy()
public_frames = frame_ids[public_mask]
assert np.all(np.diff(public_frames) == 1), "Dieser Versuch erwartet den zusammenhängenden öffentlichen Ausschnitt."
y_dead = gt["label"].eq("D").to_numpy(dtype=np.uint8)

def runs(mask):
    values = np.asarray(mask, dtype=bool)
    first = np.flatnonzero(values & ~np.r_[False, values[:-1]])
    last = np.flatnonzero(values & ~np.r_[values[1:], False])
    return np.column_stack([first, last])

dead_runs = runs(public_mask & y_dead.astype(bool))
centres = dead_runs.mean(axis=1)
cuts = np.floor((centres[:-1] + centres[1:]) / 2).astype(int)
folds = pd.DataFrame({"first": np.r_[public_frames[0], cuts + 1], "last": np.r_[cuts, public_frames[-1]]})
folds.index = pd.Index(np.arange(1, len(folds) + 1), name="fold")
split_rows = []
for fold, row in folds.iterrows():
    validation = public_mask & (frame_ids >= row["first"]) & (frame_ids <= row["last"])
    training = public_mask & ((frame_ids < row["first"] - PURGE_FRAMES) | (frame_ids > row["last"] + PURGE_FRAMES))
    training &= frame_ids % TRAIN_STRIDE == 0
    assert set(np.unique(y_dead[training])) == {0, 1}, "Ein Trainingsblock enthält nicht beide Klassen."
    contained = (dead_runs[:, 0] >= row["first"]) & (dead_runs[:, 1] <= row["last"])
    assert contained.sum() == 1, "Eine D-Phase wird durch die Blockbildung geteilt."
    split_rows.append({"fold": fold, "first": row["first"], "last": row["last"],
                       "Prüfung_s": validation.sum() / FPS, "D_Prüfung_s": y_dead[validation].sum() / FPS,
                       "Trainingszentren": training.sum(), "D_Trainingszentren": y_dead[training].sum()})
split_table = pd.DataFrame(split_rows).set_index("fold")
split_table.to_csv(OUTPUT_DIR / "folds.csv")
display(split_table.round(2))
display(gt.loc[public_mask, "label"].value_counts().rename("Öffentliche Frames").to_frame())
print(f"{len(public_frames)} öffentliche Frames, {len(dead_runs)} D-Phasen. Die vielen Frames ersetzen keine große Zahl unabhängiger Ereignisse.")

## 3. Spielerpositionen laden und Trackqualität berücksichtigen

Die Rohframe-ID minus eins ist der Videoindex. Die Ballwerte und der uneindeutige Roh-Geschwindigkeitskanal werden nicht verwendet. Wir laden ausschließlich Spieler-x/y und skalieren in Meter.

Ein Track-Slot ist keine feste Spieleridentität. Geschwindigkeiten entstehen deshalb nur bei durchgängig beobachteten Positionen im Differenzfenster; starke Sprünge werden nicht als Laufgeschwindigkeit interpretiert. Die Qualitätsgrenzen sind feste Arbeitshypothesen, keine nachträglich optimierten Parameter. Auch nach diesen Filtern können Identitätswechsel und Kameralücken Merkmale verfälschen.

In [ ]:
root = ET.parse(META_PATH).getroot()
teams = root.findall("./Metadata/Teams/Team")
side_by_team = {team.get("id"): "H" if i == 0 else "A" for i, team in enumerate(teams)}
side_by_track = {p.get("id"): side_by_team[p.get("teamId")] for p in root.findall("./Metadata/Players/Player")}
channels = {c.get("id"): c.get("playerId") for c in root.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [channels[g.find("PlayerChannelRef").get("playerChannelId")]
             for g in spec.findall("./SplitRegister/SplitRegister")]
track_sides = np.array([side_by_track[track] for track in track_ids])
player_xy = np.empty((N_FRAMES, len(track_ids), 2), dtype=np.float32)
loaded_frames = np.empty(N_FRAMES, dtype=int)
with RAW_PATH.open() as handle:
    for row, line in enumerate(handle):
        raw_frame, players_raw, _ = line.strip().split(":")
        loaded_frames[row] = int(raw_frame) - 1
        player_xy[row] = np.fromstring(players_raw.replace(";", ","), sep=",").reshape(-1, 3)[:, :2] * PITCH
assert row + 1 == N_FRAMES and np.array_equal(loaded_frames, frame_ids), "Tracking ist nicht framegleich."
player_xy[~half_mask] = np.nan
position_valid = np.isfinite(player_xy).all(axis=2)
player_xy[~position_valid] = np.nan
position_count = position_valid.sum(axis=1)
offset = DIFFERENCE_OFFSET
velocity = np.full_like(player_xy, np.nan)
velocity[offset:-offset] = (player_xy[2 * offset:] - player_xy[:-2 * offset]) * FPS / (2 * offset)
continuous_position = minimum_filter1d(position_valid.astype(np.uint8), size=2 * offset + 1,
                                     axis=0, mode="constant", cval=0).astype(bool)
speed = np.linalg.norm(velocity, axis=2)
velocity[~continuous_position | (speed > MAX_SPEED_M_S)] = np.nan
speed = np.linalg.norm(velocity, axis=2)
acceleration = np.full(speed.shape, np.nan, dtype=np.float32)
acceleration[offset:-offset] = np.linalg.norm(velocity[2 * offset:] - velocity[:-2 * offset], axis=2) * FPS / (2 * offset)
continuous_velocity = minimum_filter1d(np.isfinite(speed).astype(np.uint8), size=2 * offset + 1,
                                     axis=0, mode="constant", cval=0).astype(bool)
acceleration[~continuous_velocity | (acceleration > MAX_ACCELERATION_M_S2)] = np.nan
display(pd.DataFrame({"sichtbare Positionen": position_count[public_mask],
                      "auswertbare Geschwindigkeiten": np.isfinite(speed[public_mask]).sum(axis=1),
                      "auswertbare Beschleunigungen": np.isfinite(acceleration[public_mask]).sum(axis=1)}).describe().loc[["min", "50%", "max"]])
print(f"{len(track_ids)} Track-Slots; keine Ball-, Audio- oder Tag-Merkmale geladen.")

## 4. Austauschbare Track-Slots zu Gruppenmerkmalen verdichten

Die Merkmale hängen nicht von einer bestimmten Tracknummer ab: Geschwindigkeits- und Beschleunigungsquantile, Bewegungsanteile, räumliche Ausdehnung und Abstand des Gruppenzentrums zur Feldmitte. Für H und A ergänzen wir Anzahl, Median der Geschwindigkeit, Bewegungsanteil und Zentrumsabstände.

Die Abstände zur Feldmitte und Ausdehnungen sind bei einer 180°-Drehung gleich; wir müssen deshalb für diesen ersten Versuch keine Angriffsrichtung schätzen. Sichtbarkeit und Kameraausschnitt beeinflussen die Gruppenstatistik trotzdem. Insbesondere ist eine niedrige Bewegungsquote noch kein Beweis für D.

Fehlende Beobachtungen bleiben NaN. Die spätere Imputation lernt ihre Ersatzwerte pro Trainingsblock und fügt Missing-Indikatoren hinzu.

In [ ]:
def row_quantiles(values, quantiles):
    """Zeilenquantile mit linearer Interpolation; komplett fehlende Zeilen bleiben NaN."""
    counts = np.isfinite(values).sum(axis=1)
    ordered = np.sort(np.where(np.isfinite(values), values, np.inf), axis=1)
    ordered[counts == 0, 0] = 0
    ranks = np.maximum(counts - 1, 0)[None, :] * np.asarray(quantiles)[:, None]
    lower, upper = np.floor(ranks).astype(int), np.ceil(ranks).astype(int)
    weight = ranks - lower
    rows = np.arange(len(values))[None, :]
    result = ordered[rows, lower] * (1 - weight) + ordered[rows, upper] * weight
    result[:, counts == 0] = np.nan
    return result.astype(np.float32)

# Diese Prüfung schützt die Merkmalsberechnung besonders bei wechselnder Beobachtungszahl.
quantile_example = np.array([[1, 2, 3, 4], [1, np.nan, 5, np.nan], [np.nan] * 4])
assert np.allclose(row_quantiles(quantile_example, [.25, .5, .75]),
                   [[1.75, 2, np.nan], [2.5, 3, np.nan], [3.25, 4, np.nan]], equal_nan=True)

def group_summary(slots):
    points, velocities, accelerations = player_xy[:, slots], speed[:, slots], acceleration[:, slots]
    n = np.isfinite(points).all(axis=2).sum(axis=1)
    nv, na = np.isfinite(velocities).sum(axis=1), np.isfinite(accelerations).sum(axis=1)
    speed_q = row_quantiles(velocities, [.25, .5, .75])
    accel_q = row_quantiles(accelerations, [.5, .75])
    x_q, y_q = row_quantiles(points[:, :, 0], [.1, .9]), row_quantiles(points[:, :, 1], [.1, .9])
    centre = np.nansum(points, axis=1) / np.maximum(n[:, None], 1)
    centre[n == 0] = np.nan
    return pd.DataFrame({
        "n_position": n, "speed_coverage": nv / np.maximum(n, 1), "acceleration_coverage": na / np.maximum(n, 1),
        "speed_q25": speed_q[0], "speed_median": speed_q[1], "speed_q75": speed_q[2],
        "moving_share": np.where(nv > 0, (velocities > 2).sum(axis=1) / np.maximum(nv, 1), np.nan),
        "running_share": np.where(nv > 0, (velocities > 4).sum(axis=1) / np.maximum(nv, 1), np.nan),
        "acceleration_median": accel_q[0], "acceleration_q75": accel_q[1],
        "spread_x": (x_q[1] - x_q[0]) / PITCH[0], "spread_y": (y_q[1] - y_q[0]) / PITCH[1],
        "centre_abs_x": np.abs(centre[:, 0] - PITCH[0] / 2) / (PITCH[0] / 2),
        "centre_abs_y": np.abs(centre[:, 1] - PITCH[1] / 2) / (PITCH[1] / 2),
    }, index=pd.Index(frame_ids, name="frame"), dtype=np.float32)

base_features = group_summary(np.ones(len(track_ids), dtype=bool)).add_prefix("all_")
team_columns = ["n_position", "speed_median", "moving_share", "centre_abs_x", "centre_abs_y"]
for side in ["H", "A"]:
    base_features = base_features.join(group_summary(track_sides == side)[team_columns].add_prefix(side + "_"))
context_tables = []
for seconds in TIME_OFFSETS_SECONDS:
    delta = round(seconds * FPS)
    allowed = np.zeros(N_FRAMES, dtype=bool)
    source_frames = frame_ids + delta
    for _, first, last in HALVES:
        allowed |= half_mask & (frame_ids >= first) & (frame_ids <= last) & (source_frames >= first) & (source_frames <= last)
    shifted = base_features.shift(-delta).copy()
    shifted.loc[~allowed] = np.nan
    context_tables.append(shifted.add_suffix(f"__t{seconds:+g}s"))
features = pd.concat(context_tables, axis=1).astype(np.float32)
assert features.shape[1] == 216 and not np.isinf(features.to_numpy()).any()
display(base_features.loc[public_frames, ["all_n_position", "all_speed_median", "all_moving_share", "all_acceleration_median"]].describe().round(2))
print(f"24 Grundmerkmale × 9 Zeitpunkte = {features.shape[1]} Eingabemerkmale, {features.memory_usage().sum() / 1e6:.1f} MB.")

## 5. Vier getrennte Modellläufe pro Methode

Für jeden Prüfblock wird ein neues Modell auf den übrigen, ausreichend entfernten öffentlichen Trainingszentren angepasst. Die Klassen erhalten im Training gleiche Gesamtgewichte. Die Prüfung nutzt alle 25 Frames pro Sekunde.

Für den Medianfilter berechnen wir zusätzlich Vorhersagen im Randbereich von 45 Frames mit **demselben** Fold-Modell. Es werden keine Vorhersagen anderer Fold-Modelle in die Glättung gemischt. Der Trainingsabstand berücksichtigt Merkmalskontext, Ableitungen und diesen Filterrand. Für Trainingszentren reicht der Rohdatenkontext maximal 4,4 s; für geglättete Prüfvorhersagen 6,2 s. Der Abstand von zwölf Sekunden verhindert die Überschneidung dieser Rohdatenbereiche.

Die Modelle greifen teilweise auf spätere Zeitpunkte zu. Die Resultate gelten deshalb für Offline-Auswertung. Die Bezeichnung „out of fold“ heißt hier ausschließlich: Der jeweilige Zielblock wurde nicht für diesen Fit verwendet. Es gibt weiterhin nur ein Spiel und vier D-Ereignisse.

In [ ]:
def make_model(name):
    steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if name == "LogReg":
        steps += [("scaler", StandardScaler()),
                  ("classifier", LogisticRegression(**MODEL_CONFIG[name], random_state=SEED))]
    else:
        settings = MODEL_CONFIG[name]
        tree = DecisionTreeClassifier(max_depth=settings["max_depth"], min_samples_leaf=settings["min_samples_leaf"], random_state=SEED)
        steps += [("classifier", AdaBoostClassifier(estimator=tree, n_estimators=settings["n_estimators"],
                     learning_rate=settings["learning_rate"], random_state=SEED))]
    return Pipeline(steps)

oof = pd.DataFrame({"frame": public_frames, "GT": np.where(y_dead[public_frames], "D", "L")}).set_index("frame")
fit_records = []
for fold, row in folds.iterrows():
    first, last = int(row["first"]), int(row["last"])
    val_idx = np.arange(first, last + 1)
    train_idx = frame_ids[public_mask & ((frame_ids < first - PURGE_FRAMES) | (frame_ids > last + PURGE_FRAMES))
                          & (frame_ids % TRAIN_STRIDE == 0)]
    half = next(h for h in HALVES if h[1] <= first <= last <= h[2])
    padded_idx = np.arange(max(half[1], first - MEDIAN_FRAMES // 2), min(half[2], last + MEDIAN_FRAMES // 2) + 1)
    selected = (padded_idx >= first) & (padded_idx <= last)
    nearest_gap = np.min(np.minimum(np.abs(train_idx - first), np.abs(train_idx - last)))
    assert nearest_gap > 2 * MAX_FEATURE_RADIUS + MEDIAN_FRAMES // 2
    oof.loc[val_idx, "fold"] = fold
    weights = compute_sample_weight("balanced", y_dead[train_idx])
    for name in MODEL_CONFIG:
        model = make_model(name)
        model.fit(features.loc[train_idx], y_dead[train_idx], classifier__sample_weight=weights)
        scores = model.predict_proba(features.loc[padded_idx])[:, list(model.classes_).index(1)]
        raw = scores >= 0.5
        smoothed = median_filter(raw.astype(np.uint8), size=MEDIAN_FRAMES, mode="nearest").astype(bool)
        oof.loc[val_idx, name + "_D_score"] = scores[selected]
        oof.loc[val_idx, name + "_raw"] = np.where(raw[selected], "D", "L")
        oof.loc[val_idx, name + "_median91"] = np.where(smoothed[selected], "D", "L")
        fit_records.append({"fold": fold, "model": name, "train_samples": len(train_idx),
                            "train_D_samples": int(y_dead[train_idx].sum()), "validation_frames": len(val_idx)})
assert not oof.isna().any().any(), "Nicht alle öffentlichen Frames haben eine getrennt erzeugte Vorhersage."
oof["fold"] = oof["fold"].astype(int)
for name, prediction_path in BASELINE_PATHS.items():
    previous = pd.read_csv(prediction_path).set_index("frame")
    assert np.array_equal(previous.index, frame_ids)
    oof[name] = np.where(previous.loc[public_frames, "label"].eq("D"), "D", "L")
oof["Immer live"] = "L"
oof.to_csv(OUTPUT_DIR / "public_oof_predictions.csv", float_format="%.8f")
pd.DataFrame(fit_records).to_csv(OUTPUT_DIR / "fit_records.csv", index=False)
print(f"{len(fit_records)} Fits; alle {len(oof)} öffentlichen Frames blockweise vorhergesagt. Keine Parameter nach dem Scorevergleich angepasst.")

## 6. Frameweise und ganze Unterbrechungen vergleichen

Die binären Frame-Metriken berücksichtigen alle öffentlichen Frames und alle D-Längen. `Immer live` macht die Klassenverteilung sichtbar; Accuracy alleine ist hier leicht irreführend.

Als zusätzliche, am Paper orientierte Diagnose zählen wir D-Intervalle ab zwei Sekunden und gleichen sie bei IoU ≥ 0,5 eins zu eins ab. Die Zuordnung maximiert zuerst die Zahl gültiger Paare und danach die Überlappung. Start-/Endfehler beziehen sich nur auf gematchte Intervalle; ihre Anzahl steht daneben. Diese Diagnose ist **nicht** der offizielle Challenge-spellF1 für H/A-Phasen. Sie kann kurze Fehler verdecken und ersetzt deshalb nicht die Frame-Metriken.

Zusammengefügte Fold-Vorhersagen stammen von verschiedenen Fits. Auch die alten Methoden haben die öffentliche Referenz bereits in früheren Entwicklungsentscheidungen beeinflusst. Der Vergleich ist ein erster Machbarkeitstest, kein Ranking unabhängiger Matchmodelle.

In [ ]:
def frame_metrics(actual, predicted):
    a, p = np.asarray(actual) == "D", np.asarray(predicted) == "D"
    return {"Accuracy": accuracy_score(a, p), "Balanced_accuracy": balanced_accuracy_score(a, p),
            "D_Precision": precision_score(a, p, zero_division=0), "D_Recall": recall_score(a, p, zero_division=0),
            "D_F1": f1_score(a, p, zero_division=0), "Live_falsch_D": int((~a & p).sum())}

def stoppage_metrics(actual, predicted, name):
    true_spells, pred_spells = runs(np.asarray(actual) == "D"), runs(np.asarray(predicted) == "D")
    minimum = 2 * FPS
    true_spells = true_spells[np.diff(true_spells, axis=1).ravel() + 1 >= minimum]
    pred_spells = pred_spells[np.diff(pred_spells, axis=1).ravel() + 1 >= minimum]
    records = []
    if len(true_spells) and len(pred_spells):
        intersection = np.maximum(0, np.minimum(true_spells[:, None, 1], pred_spells[None, :, 1])
                                  - np.maximum(true_spells[:, None, 0], pred_spells[None, :, 0]) + 1)
        union = np.diff(true_spells, axis=1) + 1 + (np.diff(pred_spells, axis=1) + 1).T - intersection
        iou = intersection / union
        quality = (iou >= .5) * (min(len(true_spells), len(pred_spells)) + 1) + iou
        true_idx, pred_idx = linear_sum_assignment(-quality)
        for t, p in zip(true_idx, pred_idx):
            if iou[t, p] >= .5:
                records.append({"method": name, "GT_first": int(public_frames[0] + true_spells[t, 0]),
                                "GT_last": int(public_frames[0] + true_spells[t, 1]),
                                "pred_first": int(public_frames[0] + pred_spells[p, 0]),
                                "pred_last": int(public_frames[0] + pred_spells[p, 1]), "IoU": float(iou[t, p]),
                                "start_shift_s": (pred_spells[p, 0] - true_spells[t, 0]) / FPS,
                                "end_shift_s": (pred_spells[p, 1] - true_spells[t, 1]) / FPS})
    matched = len(records)
    denominator = len(true_spells) + len(pred_spells)
    return {"Stop_F1_IoU0.5_min2s": 2 * matched / denominator if denominator else np.nan,
            "D_Intervalle_min2s": len(pred_spells), "D_gematcht": matched,
            "Start_MAE_s_gematcht": np.mean([abs(r["start_shift_s"]) for r in records]) if records else np.nan,
            "End_MAE_s_gematcht": np.mean([abs(r["end_shift_s"]) for r in records]) if records else np.nan}, records

METHODS = ["Immer live", "Kontakt 04", "Heuristik 06", "LogReg_raw", "LogReg_median91", "AdaBoost_raw", "AdaBoost_median91"]
summary_rows, match_records, fold_rows = [], [], []
for name in METHODS:
    interval_metrics, matched = stoppage_metrics(oof["GT"], oof[name], name)
    summary_rows.append({"method": name, **frame_metrics(oof["GT"], oof[name]), **interval_metrics})
    match_records.extend(matched)
    for fold, local in oof.groupby("fold"):
        fold_rows.append({"fold": fold, "method": name, **frame_metrics(local["GT"], local[name])})
results = pd.DataFrame(summary_rows).set_index("method")
fold_results = pd.DataFrame(fold_rows)
results.to_csv(OUTPUT_DIR / "public_oof_metrics.csv")
fold_results.to_csv(OUTPUT_DIR / "public_fold_metrics.csv", index=False)
pd.DataFrame(match_records).to_csv(OUTPUT_DIR / "public_matched_stoppages.csv", index=False)
display(results[["D_Precision", "D_Recall", "D_F1", "Balanced_accuracy", "Stop_F1_IoU0.5_min2s", "D_Intervalle_min2s", "D_gematcht"]].round(4))
display(fold_results.pivot(index="method", columns="fold", values="D_F1").loc[METHODS].round(3))
print("Keine Methode anhand dieser Tabelle nachoptimiert oder automatisch für die Abgabe ausgewählt.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
results[["D_Precision", "D_Recall", "D_F1"]].plot.bar(ax=axes[0], rot=24, color=["#a5bfd3", "#6497b8", "#245e93"])
axes[0].set(title="Öffentliche D-Erkennung: blockweise getrennte ML-Prüfung", xlabel="", ylabel="Score", ylim=(0, 1))
learned = [name for name in METHODS if name.startswith(("LogReg", "AdaBoost"))]
fold_table = fold_results.pivot(index="fold", columns="method", values="D_F1")[learned]
for name in learned:
    axes[1].plot(fold_table.index, fold_table[name], marker="o", label=name)
axes[1].set(title="Eine Unterbrechung pro Prüfblock: schwankende Leistung", xlabel="Prüfblock", ylabel="D-F1", ylim=(0, 1), xticks=folds.index)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(axis="y", alpha=.15)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
relative_s = (public_frames - public_frames[0]) / FPS
timeline_names = ["GT", "Kontakt 04", "Heuristik 06", "LogReg_median91", "AdaBoost_median91"]
raster = np.array([oof[name].eq("D").to_numpy(dtype=int) for name in timeline_names])
axes[0].imshow(raster, interpolation="nearest", aspect="auto", cmap=ListedColormap(["#32816a", "#999999"]),
               vmin=0, vmax=1, extent=(0, len(oof) / FPS, len(timeline_names) - .5, -.5))
axes[0].set(yticks=range(len(timeline_names)), yticklabels=timeline_names, title="Referenz und D/Live-Spuren im öffentlichen Ausschnitt")
fig.legend(handles=[Patch(color="#32816a", label="L · Spiel läuft"), Patch(color="#999999", label="D · unterbrochen")], loc="outside upper right", ncol=2)
for name, ax in zip(MODEL_CONFIG, axes[1:]):
    ax.plot(relative_s, oof[name + "_D_score"], color="#245e93", linewidth=.9)
    ax.axhline(.5, color="0.5", linestyle="--", linewidth=.8)
    ax.set(ylabel=name + " D-Score", ylim=(0, 1))
    for first, last in dead_runs:
        ax.axvspan((first - public_frames[0]) / FPS, (last + 1 - public_frames[0]) / FPS, color="#999999", alpha=.2)
for cut in cuts:
    for ax in axes:
        ax.axvline((cut + 1 - public_frames[0]) / FPS, color="#c88c35", linestyle=":", linewidth=1)
axes[-1].set_xlabel("Sekunden ab Beginn der öffentlichen Referenz; gepunktet: Wechsel des Fold-Modells")
plt.show()

## 7. Beide Modelle mit allen öffentlichen Trainingslabels einfrieren

Nach der getrennten Prüfung passen wir beide festgelegten Modelle mit allen öffentlichen Trainingszentren an. Diese Fits sind für spätere Szenen gedacht. Ihre Vorhersagen **im öffentlichen Trainingsausschnitt** sind keine unabhängige Evaluation und werden nicht als neuer Testscore ausgegeben.

Die Vollspiel-Diagnose enthält D/L nur in den veröffentlichten Halbzeitbereichen, außerhalb X. Der Medianfilter wird je Halbzeit separat angewandt. Beide Modellvarianten werden gespeichert; es gibt keine Auswahl des besseren Modells anhand laufender Annotationen. Die Dateien sind Diagnose- und Statusvorhersagen, keine H/A/D-Abgabe.

In [ ]:
full_train_idx = public_frames[public_frames % TRAIN_STRIDE == 0]
weights = compute_sample_weight("balanced", y_dead[full_train_idx])
final_models = {}
full_diagnostics = pd.DataFrame({"frame": frame_ids, "public_training_region": public_mask,
                                 "visible_player_positions": position_count})
active_frames = frame_ids[half_mask]
for name in MODEL_CONFIG:
    model = make_model(name)
    model.fit(features.loc[full_train_idx], y_dead[full_train_idx], classifier__sample_weight=weights)
    scores = np.full(N_FRAMES, np.nan, dtype=np.float32)
    for first in range(0, len(active_frames), 20000):
        idx = active_frames[first:first + 20000]
        scores[idx] = model.predict_proba(features.loc[idx])[:, list(model.classes_).index(1)]
    raw_labels = np.full(N_FRAMES, "X", dtype="<U1")
    raw_labels[half_mask] = np.where(scores[half_mask] >= .5, "D", "L")
    smooth_labels = raw_labels.copy()
    for _, first, last in HALVES:
        smooth_dead = median_filter((raw_labels[first:last + 1] == "D").astype(np.uint8), size=MEDIAN_FRAMES, mode="nearest")
        smooth_labels[first:last + 1] = np.where(smooth_dead, "D", "L")
    full_diagnostics[name + "_D_score"] = scores
    full_diagnostics[name + "_raw"] = raw_labels
    full_diagnostics[name + "_median91"] = smooth_labels
    final_models[name] = model
    joblib.dump(model, MODEL_DIR / f"{name.lower()}_v1.joblib")
full_diagnostics.to_csv(OUTPUT_DIR / "full_match_status_diagnostics.csv", index=False, float_format="%.6f")
(OUTPUT_DIR / "feature_columns.json").write_text(json.dumps(list(features.columns), indent=2), encoding="utf-8")
print(f"Zwei Modelle auf {len(full_train_idx)} Trainingszentren angepasst und gespeichert. Öffentliche Trainingsframes im Vollspiel-Export ausdrücklich markiert.")

## 8. Vorhersagen für die Kontrollclips ohne Zugriff auf Annotationen sichern

Wir lesen nur die bereits festgelegten Szenengrenzen aus Notebook 06. Die Referenzdateien und die Annotationsansicht aus Notebook 07 werden nicht geöffnet. Diese Vorhersagen können nach Abschluss der menschlichen Annotation gegen die bestätigten D/L-Abschnitte verglichen werden; U bleibt dann ausgeschlossen.

Die Datei `control_predictions_frozen_v1.csv` wird bei erneutem Ausführen nur akzeptiert, wenn sie bytegleich bleibt. Für eine methodisch andere Variante muss eine neue Versionsdatei entstehen. So bleibt der erste Vergleich nachvollziehbar, auch wenn spätere Annotationen zu Änderungen führen.

In [ ]:
control_scenes = pd.read_csv(SCENES_PATH).set_index("scene_id").loc[
    ["control_free_kick", "control_goal_kick", "control_throw_in"]]
control_tables = []
for scene_id, scene in control_scenes.iterrows():
    idx = np.arange(int(scene["first"]), int(scene["last"]) + 1)
    assert not public_mask[idx].any(), "Ein Kontrollclip liegt in den öffentlichen Trainingsdaten."
    local = full_diagnostics.iloc[idx].copy()
    local.insert(0, "scene_id", scene_id)
    control_tables.append(local)
control_predictions = pd.concat(control_tables, ignore_index=True)
frozen_path = OUTPUT_DIR / "control_predictions_frozen_v1.csv"
frozen_bytes = control_predictions.to_csv(index=False, float_format="%.6f", lineterminator="\n").encode("utf-8")
if frozen_path.exists():
    assert frozen_path.read_bytes() == frozen_bytes, "Die eingefrorene v1 würde sich ändern. Eine neue Modellvariante benötigt eine neue Versionsdatei."
else:
    frozen_path.write_bytes(frozen_bytes)
frozen_sha256 = hashlib.sha256(frozen_bytes).hexdigest()
print(f"{len(control_predictions)} Kontrollframes ohne Referenzlabels vorhergesagt und eingefroren.")
print(f"SHA256: {frozen_sha256}")
display(control_predictions.groupby("scene_id")[[name + "_median91" for name in MODEL_CONFIG]].agg(lambda s: round(100 * s.eq("D").mean(), 1)).rename(columns=lambda c: c + " D-Anteil (%)"))

## 9. Welche Hinweise nutzt der abschließende AdaBoost-Fit?

Die Wichtigkeiten gehören zum Fit auf allen öffentlichen Trainingszentren. Sie erklären weder kausale Zusammenhänge noch die Leistung auf neuen Spielen. Bei so wenigen Unterbrechungen kann eine hohe Wichtigkeit auch einen spezifischen Kameraausschnitt oder fehlende Beobachtungen widerspiegeln. Der Plot soll solche Abhängigkeiten sichtbar machen.

Wir gruppieren die Wichtigkeiten nach Grundmerkmal und nach Zeitversatz. Missing-Indikatoren werden mit ihrem zugehörigen Merkmal zusammengeführt.

In [ ]:
ada = final_models["AdaBoost"]
transformed_names = ada.named_steps["imputer"].get_feature_names_out(features.columns)
importance = pd.DataFrame({"feature": transformed_names, "importance": ada.named_steps["classifier"].feature_importances_})
importance["base"] = importance["feature"].str.replace("missingindicator_", "", regex=False).str.split("__t").str[0]
importance["time_s"] = importance["feature"].str.split("__t").str[-1].str.removesuffix("s").astype(float)
importance.to_csv(OUTPUT_DIR / "adaboost_feature_importance.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
importance.groupby("base")["importance"].sum().nlargest(10).sort_values().plot.barh(ax=axes[0], color="#6497b8")
axes[0].set(title="AdaBoost: zehn stärkste Merkmalsgruppen", xlabel="Summe der Modellwichtigkeit", ylabel="")
importance.groupby("time_s")["importance"].sum().sort_index().plot.bar(ax=axes[1], color="#6497b8", rot=0)
axes[1].set(title="AdaBoost: Wichtigkeit nach Zeitversatz", xlabel="Sekunden relativ zum Vorhersageframe", ylabel="Summe der Modellwichtigkeit")
plt.show()

## 10. Ergebnis und Grenzen des ersten Tests

Die folgenden Werte stammen ausschließlich aus der blockweise getrennten öffentlichen Prüfung. Eine Übereinstimmung mit den Paper-Ergebnissen wird nicht erwartet: Dort wurden 45 Spiele zum Training und 47 andere Spiele zum Test verwendet, hier nur vier D-Phasen desselben Spiels.

Der Originalansatz mit geordneten 22-Spieler-Eingaben und langen Zeitversätzen bis ±80 s wurde nicht reproduziert. Die Reduktion auf Gruppenstatistik und ±4 s, die Klassenwichtung und die Qualitätsfilter sind eigene Anpassungen. Auch perfekte Ergebnisse auf einzelnen Blöcken wären kein Nachweis für Gesamtspielleistung.

Die nächsten Schritte hängen vom späteren Vergleich mit bestätigten Kontrolllabels ab. Die bereits gesichteten Clips sind keine unberührte Testmenge; vor weiterer Optimierung sollten zusätzliche, bisher unangesehene Szenen festgelegt werden.

In [ ]:
for name in MODEL_CONFIG:
    raw, smooth = results.loc[name + "_raw"], results.loc[name + "_median91"]
    display(Markdown(f"**{name}, getrennte öffentliche Prüfung:** D-F1 roh **{raw['D_F1']:.3f}**, "
                     f"mit festem Medianfilter **{smooth['D_F1']:.3f}**; "
                     f"D-Precision/Recall geglättet **{smooth['D_Precision']:.3f}/{smooth['D_Recall']:.3f}**. "
                     f"Gematchte Unterbrechungen ab 2 s: **{int(smooth['D_gematcht'])}/{len(dead_runs)}**."))
unchanged = {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest() == digest for name, digest in INPUT_HASHES.items()}
assert all(unchanged.values()), "Eine bisherige eingefrorene Eingabe wurde verändert."
display(pd.Series(unchanged, name="Bisherige Eingabe unverändert").to_frame())
manifest = {"experiment": "player_status_ml_v1", "paper_doi": "10.1038/s41598-022-19948-1", "exact_reproduction": False,
    "feature_design": "24 permutation-invariant player/group statistics at 9 time offsets", "n_input_features": features.shape[1],
    "time_offsets_seconds": TIME_OFFSETS_SECONDS, "difference_offset_frames": DIFFERENCE_OFFSET,
    "max_speed_m_s": MAX_SPEED_M_S, "max_acceleration_m_s2": MAX_ACCELERATION_M_S2,
    "fps": FPS, "n_frames": N_FRAMES, "seed": SEED, "train_stride_frames": TRAIN_STRIDE,
    "purge_frames": PURGE_FRAMES, "feature_radius_frames": MAX_FEATURE_RADIUS, "median_frames": MEDIAN_FRAMES,
    "models": MODEL_CONFIG, "class_weighting": "balanced sample weights in classifier fit", "threshold": .5,
    "parameter_search": False, "offline_uses_future_features": True,
    "public_reference_use": "training and within-match purged block CV; previously seen in earlier experiments",
    "folds": folds.reset_index().to_dict("records"), "n_public_D_episodes": len(dead_runs),
    "input_sha256": INPUT_HASHES, "control_predictions_sha256": frozen_sha256,
    "annotation_files_read": False, "annotation_files_written": False, "submission_exported": False,
    "python": sys.version.split()[0], "packages": {name: version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]}}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Ergebnisse, Modelle und eingefrorene Kontrollvorhersagen: {OUTPUT_DIR}")

### Einordnung des ersten v1-Laufs

AdaBoost mit Medianfilter erzielt **D-F1 0,756**, gegenüber **0,797** für die bisherige Heuristik 06. Es markiert weniger laufende Frames fälschlich als D (**129 statt 285**) und hat höhere D-Precision (**0,883 statt 0,803**), erkennt aber weniger tatsächliche D-Frames (**Recall 0,661 statt 0,791**). Das ist ein anderer Fehlertausch, keine insgesamt nachgewiesene Verbesserung.

Die getrennten Blöcke zeigen den kleinen Datenumfang deutlich: Beim AdaBoost-Medianmodell reicht D-F1 von **0,292 bis 0,874**. Besonders schwierig ist die D-Phase **18027–18388** im vierten Prüfblock; nur **17,1 %** ihrer D-Frames werden erkannt. Der Roh-AdaBoost hat dort denselben Recall, der Medianfilter alleine erklärt diese Auslassung also nicht. Die genaue Ursache ist mit späteren Szenen und bestätigten Labels zu untersuchen.

Die logistische Regression ist in diesem Versuch schwächer; der fest übernommene Medianfilter senkt ihren D-F1 von **0,629 auf 0,613**. Eine Glättung ist damit kein automatisch hilfreicher Schritt. Bei AdaBoost verbessert sie den D-F1 von **0,744 auf 0,756** und die Zahl gematchter D-Intervalle von zwei auf drei.

**Arbeitsentscheidung:** Beide Modelle als eingefrorene erste Varianten behalten, vorerst keine Heuristik ersetzen und keine Schwellen nach diesem kleinen Vergleich nachstellen. Für die drei Kontrollclips liegen bereits Vorhersagen vor. Sie werden erst nach Abschluss der menschlichen Annotation gegen die bestätigten D/L-Abschnitte geprüft.

**Danach gemeinsam prüfen:** Sobald die Annotationen fertig sind, die eingefrorenen Kontrollvorhersagen mit den menschlich bestätigten D/L-Abschnitten aus Notebook 07 vergleichen. Erst danach entscheiden, ob Spielstatusmerkmale, Glättung oder die Einbindung in die H/A-Pipeline geändert werden sollen. Dieses Notebook trainiert keine Teamzuordnung und erzeugt keine neue Challenge-Abgabe.